# HTML and Webscraping

In [76]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

## Scraping an HTML table

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2025). We'll use Beautiful Soup to scrape information from this table.

1\. Read in the HTML from the URL using the `requests` library.

In [ ]:
link = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
headers = {"User-Agent": "GSB5544 practice activity"}
wikipedia = requests.get(link, headers=headers)
wikipedia.status_code

200

2\. Use Beautiful Soup to parse this string into a tree called `soup`

In [ ]:
soup = BeautifulSoup(wikipedia.text, "html.parser")

3\. Determine how many tables are in `soup`. (Hint: use `find_all("table")`.)

In [ ]:
len(soup.find_all("table"))

10

4\. There are several tables included in `soup`, so we need to narrow it down. Go to the cities table Wikipedia page and "Inspect" it. What are the attributes (class, style) of this table?

**class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center jquery-tablesorter"**

**style="text-align:right"**



5\. You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center" style="text-align:right">
```

How many tables in `soup` have these attributes?

In [80]:
len(soup.find_all("table", attrs={"class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center", "style": "text-align:right"}))

1

6\. There should only be 1 table of this type, so we just need to select it. The following code finds all tables with the desired attributes and then selects the first (only) one to store as `table`. (You just need to run this.)

In [81]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

7\. Our goal is now to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for:

- city
- state
- population (2025 estimate)
- 2020 land area (sq mi).

First, let's just see how to scrape the information for New York City. Starting from `table` create an object, called `city`, that contains the information just for New York City.

Hints: Inspect the source; what kind of tag represents each row? Find all tags of this type in `table` and select the first one that corresponds to a city. Note that the first 3 rows of the table are headers.

In [82]:
records = table.find_all("tr")
nyc = records[3]
city = nyc.find_all("td")
city[0].text, city[1].text, city[2].text, city[5].text

('New York[c]', 'NY', '8,584,629', '300.5')

8\. Starting with `city` extract the city's name and store it as `name`.

Hints: Inspect the source; what tag represents the cells within a row? Find all tags of this type and extract the text corresponding to the cell with the city's name.

In [83]:
city_record = []
city_record.append({"name": city[0].text})

9\. Extract the city's state and store it as `state`.

In [84]:
city_record.append({"state": city[1].text})

10\. Extract the city's population and store is as `population`.

In [85]:
city_record.append({"population": city[2].text})

11\. Extract the city's area and store is as `area`.

In [86]:
city_record.append({"area": city[5].text})

12\. Now put the steps for a single city into a loop to extract the information for all cities in `table` and create a data frame.

Hints:
- Start with an empty list named `rows`
- Write a loop that starts `for city in ...` and replace `...` code that finds all the table rows. (Use what you did in part 7, but don't just select one row. Select all rows except for the 3 header rows.)
- Use your code from 8-11 to extract the information for the city
- And append it to `rows` as "name", "state", "population", "area"
- Convert `rows` into a Pandas data frame. You should obtain a data frame with 348 rows and 4 columns.


In [91]:
import re

rows = []
for row in records[3:]:
    cells = row.find_all("td")
    rows.append({
        "name": re.sub(r"\[.*?\]", "", cells[0].text),
        "state": cells[1].text,
        "population": cells[2].text,
        "area": int(cells[3].text.replace(",","")),
    })

df = pd.DataFrame(rows)
df

,name,state,population,area
0,New York,NY,"8,584,629",8804190
1,Los Angeles,CA,"3,869,089",3898747
2,Chicago,IL,"2,731,585",2746388
3,Houston,TX,"2,397,315",2304580
4,Phoenix,AZ,"1,665,481",1608139
...,...,...,...,...
343,San Angelo,TX,"100,640",99893
344,Edmond,OK,"100,479",94428
345,Davenport,IA,"100,358",101724
346,Deltona,FL,"100,267",93692


13\. Use the Pandas command `pd.read_html` can be used to scrape the table from the webpage. Note: `read_html` will return all the tables, so you will need to narrow your request using attributes. You don't need to worry about selecting columns; just scrape the whole table.

In [97]:
tables = pd.read_html(
    StringIO(wikipedia.text),
    attrs={
        "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
        "style": "text-align:right"
    }
)

## Scraping from multiple webpages

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/. Notice that the information is spread over many pages.

1\. Scrape the information from the first page with Beatiful Soup.

In [99]:
hockeylink = "https://www.scrapethissite.com/pages/forms"
hockey = requests.get(hockeylink, headers=headers)
hockeysoup = BeautifulSoup(hockey.text, "html.parser")

2\. Find the main table on this page and store it as `table`.

In [100]:
table = hockeysoup.find_all("table",
                  attrs={
                      "class": "table"}
                  )[0]

3\. Extract the information from the cells of this table into a Pandas data frame.

In [102]:
column_names = [
    th.get_text(strip=True)
    for th in table.find("tr").find_all("th")
]

rows = []

for row in table.find_all("tr")[1:]:
    cells = [
        td.get_text(strip=True)
        for td in row.find_all("td")
    ]

    rows.append(dict(zip(column_names, cells)))

df = pd.DataFrame(rows)

df

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25
5,Edmonton Oilers,1990,37,37,,0.463,272,272,0
6,Hartford Whalers,1990,31,38,,0.388,238,276,-38
7,Los Angeles Kings,1990,46,24,,0.575,340,254,86
8,Minnesota North Stars,1990,27,39,,0.338,256,266,-10
9,Montreal Canadiens,1990,39,30,,0.487,273,249,24


4\. But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We could switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [104]:
pagination = hockeysoup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [105]:
for link in links:
  print(link.attrs["href"])

/pages/forms/?page_num=1
/pages/forms/?page_num=2
/pages/forms/?page_num=3
/pages/forms/?page_num=4
/pages/forms/?page_num=5
/pages/forms/?page_num=6
/pages/forms/?page_num=7
/pages/forms/?page_num=8
/pages/forms/?page_num=9
/pages/forms/?page_num=10
/pages/forms/?page_num=11
/pages/forms/?page_num=12
/pages/forms/?page_num=13
/pages/forms/?page_num=14
/pages/forms/?page_num=15
/pages/forms/?page_num=16
/pages/forms/?page_num=17
/pages/forms/?page_num=18
/pages/forms/?page_num=19
/pages/forms/?page_num=20
/pages/forms/?page_num=21
/pages/forms/?page_num=22
/pages/forms/?page_num=23
/pages/forms/?page_num=24
/pages/forms/?page_num=1


Now we can loop over `links` to make a request to the url for each page and scrape the data into a table similar to what we did for the first page. Write such a loop to extract the data and create a Pandas data frame.


A few technicalities:

- You might need to skip the "previous" and "next" buttons
- So you don't keep repeating headers, you will want to skip rows that don't represent teams.

In [113]:
column_names = [
    th.get_text(strip=True)
    for th in table.find("tr").find_all("th")
]
rows = []
nums = range(1, 25, 1)

for num in nums:
    page = f"https://www.scrapethissite.com/pages/forms/?page_num={num}"
    hockey = requests.get(page, headers=headers)
    hockeysoup = BeautifulSoup(hockey.text, "html.parser")
    table = hockeysoup.find_all("table",
                  attrs={
                      "class": "table"}
                  )[0]
    for row in table.find_all("tr")[1:]:
        cells = [
            td.get_text(strip=True)
            for td in row.find_all("td")
        ]

        rows.append(dict(zip(column_names, cells)))

df = pd.DataFrame(rows)

df

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25
...,...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9,0.622,249,198,51
580,Washington Capitals,2011,42,32,8,0.512,222,230,-8
